In [1]:
!pip install trl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 518.9/518.9 kB 11.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 111.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 42.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 80.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 116.4 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 90.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import os
import json
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import GRPOConfig, GRPOTrainer  # pip install trl>=0.9.0
from dataclasses import dataclass
from typing import Dict, List
import random
import copy
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BASE_NAME = "deepseek-ai/deepseek-coder-1.3b-instruct"

tokenizer = AutoTokenizer.from_pretrained(BASE_NAME, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

# ---- MBPP dataset ----
MBPP_PATH = "/kaggle/input/asemjson/mbpp.jsonl" 

def load_mbpp(path: str):
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            ex = json.loads(line)
            data.append(ex)
    return data

mbpp_all = load_mbpp(MBPP_PATH)
random.seed(42)
random.shuffle(mbpp_all)

train_data = mbpp_all[:350]
val_data   = mbpp_all[350:400]
test_data  = mbpp_all[400:500]

2025-12-27 09:03:44.270338: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766826224.654590      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766826224.761437      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/631 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.69G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

In [3]:
CODE_GEN_TEMPLATE = """You are an AI programming assistant, utilizing the
DeepSeek Coder model, developed by DeepSeek Company,
and you only answer questions related to computer science.

### Instruction:
Write a Python function that solves the following problem:
{task_description}

Required function signature:
{signature}

Requirements:
- Implement only the core function
- Include necessary import statements
- Ensure the function signature matches the test cases
- Write clean, efficient code

### Response:
"""

def extract_signature(code: str) -> str:
    for line in code.splitlines():
        line = line.strip()
        if line.startswith("def "):
            return line  # например: "def sum_Even(l, r):"
    return ""  # на всякий случай

@dataclass
class MbppRLExample:
    task_id: int
    prompt: str
    test_code: str
    ref_code: str

class MbppRLDataset(Dataset):
    def __init__(self, examples: List[Dict]):
        self.examples = []
        for ex in examples:
            task_desc = ex["text"]
            tests = "\n".join(ex["test_list"])
            ref = ex["code"]
            signature = extract_signature(ref)

            prompt = CODE_GEN_TEMPLATE.format(
                task_description=task_desc,
                signature=signature,
            )

            self.examples.append(
                MbppRLExample(
                    task_id=ex.get("task_id", 0),
                    prompt=prompt,
                    test_code=tests,
                    ref_code=ref,
                )
            )

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ex = self.examples[idx]
        return {
            "prompt": ex.prompt,
            "test_code": ex.test_code,
            "task_id": ex.task_id,
        }

train_dataset = MbppRLDataset(train_data)
val_dataset   = MbppRLDataset(val_data)
test_dataset   = MbppRLDataset(test_data)

In [4]:
def extract_code(completion: str) -> str:
    """
    Выделяет пригодный к выполнению Python-код из completion'а модели.
    Стратегия:
    1) Если есть блоки с ``````, берём тот, где есть 'def '.
    2) Иначе берём всё от первой строки, начинающейся с 'def '.
    3) fallback — исходный completion.
    """
    lines = completion.splitlines()

    # 1. Ищем блоки ``````
    if "```" in completion:
        parts = completion.split("```")
        for part in parts:
            if "def " in part:
                # убираем возможный язык после ```
                part_lines = part.splitlines()
                # отбрасываем первую строку, если это 'python' или пустота
                if part_lines and part_lines[0].strip().lower() in ("python", ""):
                    part_lines = part_lines[1:]
                return "\n".join(part_lines).strip()

    # 2. Иначе берём всё от первой строки с def
    for i, line in enumerate(lines):
        if line.strip().startswith("def "):
            return "\n".join(lines[i:]).strip()

    # 3. fallback
    return completion.strip()

In [5]:
import subprocess
import tempfile
import textwrap


def run_python_with_tests(solution_code: str, test_code: str, timeout: int = 5):
    """
    Выполняет solution_code + test_code в отдельном питоновском процессе и
    возвращает число прошедших и общее число тестов.
    Предполагается, что test_code использует assert'ы.
    """
    script = solution_code + "\n\n" + test_code
    with tempfile.NamedTemporaryFile(mode="w", suffix=".py", delete=False) as f:
        f.write(script)
        tmp_path = f.name

    try:
        proc = subprocess.run(
            ["python", tmp_path],
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            timeout=timeout,
            text=True,
        )
        # Грубая эвристика: считаем, что все assert прошли, если код возврата 0
        # и общее число тестов берём по количеству строк с "assert".
        total_tests = sum(1 for line in test_code.splitlines() if "assert" in line)
        passed = total_tests if proc.returncode == 0 else 0
    except subprocess.TimeoutExpired:
        passed, total_tests = 0, sum(1 for line in test_code.splitlines() if "assert" in line)
    finally:
        os.remove(tmp_path)

    return passed, total_tests

def reward_fn_test(prompts, completions, **kwargs):

    test_codes_batch = kwargs["test_code"]   # список строк длины batch_size
    task_ids_batch   = kwargs["task_id"]     # список int длины batch_size

    batch_size = len(prompts)
    num_generations = len(completions) // batch_size  # K

    expanded_test_codes = []
    expanded_task_ids = []
    for tcode, tid in zip(test_codes_batch, task_ids_batch):
        expanded_test_codes.extend([tcode] * num_generations)
        expanded_task_ids.extend([tid] * num_generations)

    rewards = []
    for completion, tid, tcode in zip(completions, expanded_task_ids, expanded_test_codes):
        code = extract_code(completion)
        passed, total = run_python_with_tests(code, tcode)
        r = passed / total if total > 0 else 0.0
        rewards.append(float(r))
    #print(rewards)
    return rewards 


In [6]:
import os
os.environ["WANDB_MODE"] = "disabled"
os.environ["WANDB_DISABLED"] = "true"
K = 4  # число генераций на задачу

grpo_config_test = GRPOConfig(
    do_train=True,
    do_eval=True,                     # мониторим валид. set по R_test
    learning_rate=3e-5,               # в диапазоне 2e-5–5e-5 из методологии
    per_device_train_batch_size=1,    # 1–2 задачи на шаг
    per_device_eval_batch_size=1,
    num_generations=K,
    generation_batch_size=K,          # кратно num_generations
    num_train_epochs=1,
    max_steps=300,                    # 500–1000 апдейтов
    logging_steps=10,
    save_steps=10e6,
    save_total_limit=0,
    #output_dir="grpo-testonly",
    remove_unused_columns=False,
    report_to=[],                     # отключаем wandb
    generation_kwargs={
        "max_new_tokens": 256,
        "temperature": 0.7,
        "top_p": 0.9,
        "do_sample": True,
    },
)

trainer_test = GRPOTrainer(
    model=model,              # DeepSeekCoder-1.3B-Instruct (policy)
    reward_funcs=reward_fn_test,
    args=grpo_config_test,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)

trainer_test.train()
#trainer_test.save_model("deepseek-grpo-unit-tests")
#tokenizer.save_pretrained("deepseek-grpo-unit-tests")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 32021}.


Step,Training Loss
10,0.000000
20,-0.001800
30,-0.025000
40,0.025600
50,0.000000
60,0.000900
70,0.029200
80,-0.023400
90,0.004400
100,0.001500


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


TrainOutput(global_step=300, training_loss=0.001042229856053988, metrics={'train_runtime': 2531.9125, 'train_samples_per_second': 0.118, 'train_steps_per_second': 0.118, 'total_flos': 0.0, 'train_loss': 0.001042229856053988})

In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

JUDGE_MODEL_NAME = "flowaicom/Flow-Judge-v0.1"

judge_tokenizer = AutoTokenizer.from_pretrained(JUDGE_MODEL_NAME, use_fast=True)
judge_tokenizer.pad_token = judge_tokenizer.eos_token

judge_model = AutoModelForCausalLM.from_pretrained(
    JUDGE_MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
judge_model.eval()

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/569 [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/2.67G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/172 [00:00<?, ?B/s]

Phi3ForCausalLM(
  (model): Phi3Model(
    (embed_tokens): Embedding(32064, 3072, padding_idx=32000)
    (layers): ModuleList(
      (0-31): 32 x Phi3DecoderLayer(
        (self_attn): Phi3Attention(
          (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
          (qkv_proj): Linear(in_features=3072, out_features=9216, bias=False)
        )
        (mlp): Phi3MLP(
          (gate_up_proj): Linear(in_features=3072, out_features=16384, bias=False)
          (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
          (activation_fn): SiLUActivation()
        )
        (input_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
        (resid_attn_dropout): Dropout(p=0.0, inplace=False)
        (resid_mlp_dropout): Dropout(p=0.0, inplace=False)
      )
    )
    (norm): Phi3RMSNorm((3072,), eps=1e-05)
    (rotary_emb): Phi3RotaryEmbedding()
  )
  (lm_head): Linear(in_features=3072, out_featur

In [7]:
import re
from typing import List, Dict

JUDGE_PROMPT_TEMPLATE = """
You are an expert code reviewer evaluating the quality of Python solutions.
Analyze the provided code based on the following criteria:

Problem Statement:
{problem_description}

Generated Code:
{generated_code}

Evaluation Criteria:
1. Correctness: Does the code correctly solve the stated problem?
2. Code Quality: Is the code readable, well-structured, and following
   Python best practices?
3. Efficiency: Are algorithms and data structures chosen appropriately?
4. Completeness: Does the solution handle edge cases and include
   necessary imports?

Return ONLY the final score as floating point between 0.0 and 1.0
"""

@torch.no_grad()
def call_local_judge(problem_description: str, generated_code: str) -> float:
    prompt = JUDGE_PROMPT_TEMPLATE.format(
        problem_description=problem_description,
        generated_code=generated_code,
    )

    inputs = judge_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048,
    ).to(judge_model.device)

    output_ids = judge_model.generate(
        **inputs,
        max_new_tokens=32,
        do_sample=False,
        #temperature=0.0,
    )

    full_text = judge_tokenizer.decode(output_ids[0], skip_special_tokens=True)
    generated = full_text[len(prompt):].strip()

    # ищем первое число вида 0.x, 1 или 1.0
    match = re.search(r"([01](?:\.\d+)?)", generated)
    if match:
        score = float(match.group(1))
        score = max(0.0, min(1.0, score))
    else:
        score = 0.0

    return score

In [8]:
def reward_llm_grpo_fn(prompts, completions, **kwargs):
    """
    prompts:     список исходных промптов (len = batch_size)
    completions: список сгенерированных ответов (len = batch_size * K)
    kwargs:      содержит те же метаданные, что и в датасете:
                 например, "task_description" или "prompt" целиком.
    """
    # достаём описание задачи; можно хранить отдельно, но можно и так:
    # здесь предполагаем, что в kwargs есть исходные тексты задач
    # если их нет — надо положить их в __getitem__ датасета,
    # например "task_description": ex.original_text
    task_desc_batch = kwargs.get("task_description", prompts)  # len = batch_size

    batch_size = len(prompts)
    num_generations = len(completions) // batch_size  # K

    # размножаем описания задач по числу генераций
    expanded_desc = []
    for desc in task_desc_batch:
        expanded_desc.extend([desc] * num_generations)

    rewards = []
    for raw_completion, desc in zip(completions, expanded_desc):
        code = extract_code(raw_completion)          # тот же extract_code, что для тестов
        r = call_local_judge(desc, code)            # R_LLM \in [0,1]
        rewards.append(float(r))

    return rewards

In [ ]:
# import os
os.environ["WANDB_MODE"] = "disabled"
os.environ["WANDB_DISABLED"] = "true"
K = 6  # число генераций на задачу

#DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

grpo_config_llm = GRPOConfig(
    do_train=True,
    do_eval=True,                     # мониторим валид. set по R_test
    learning_rate=3e-5,               # в диапазоне 2e-5–5e-5 из методологии
    per_device_train_batch_size=1,    # 1–2 задачи на шаг
    per_device_eval_batch_size=1,
    num_generations=K,
    generation_batch_size=K,          # кратно num_generations
    num_train_epochs=1,
    max_steps=300,                    # 500–1000 апдейтов
    logging_steps=10,
    save_steps=150,
    save_total_limit=2,
    #output_dir="grpo-testonly",
    remove_unused_columns=False,
    report_to=[],                     # отключаем wandb
    generation_kwargs={
        "max_new_tokens": 256,
        "temperature": 0.7,
        "top_p": 0.9,
        "do_sample": True,
    },
)

trainer_llm = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    args=grpo_config_llm,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    reward_funcs=reward_llm_grpo_fn,
)

trainer_llm.train()
#trainer_llm.save_model("deepseek-grpo-llmonly")
#llm_tokenizer.save_pretrained("deepseek-grpo-llmonly")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 32021}.


Step,Training Loss
10,-0.002000
20,-0.000400
30,0.010300


In [9]:
#A_UT  = 0.503
#B_LLM = 0.497

A_UT  = 0.7
B_LLM = 0.3

def reward_fn_combined(prompts, completions, **kwargs):
    r_test = reward_fn_test(prompts, completions, **kwargs)
    r_llm  = reward_llm_grpo_fn(prompts, completions, **kwargs)
    return [A_UT * rt + B_LLM * rl for rt, rl in zip(r_test, r_llm)]

In [10]:
# import os
os.environ["WANDB_MODE"] = "disabled"
os.environ["WANDB_DISABLED"] = "true"
K = 4  # число генераций на задачу

#DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

grpo_config_ful = GRPOConfig(
    do_train=True,
    do_eval=True,                     # мониторим валид. set по R_test
    learning_rate=3e-5,               # в диапазоне 2e-5–5e-5 из методологии
    per_device_train_batch_size=1,    # 1–2 задачи на шаг
    per_device_eval_batch_size=1,
    num_generations=K,
    generation_batch_size=K,          # кратно num_generations
    num_train_epochs=1,
    max_steps=300,                    # 500–1000 апдейтов
    logging_steps=10,
    save_steps=200,
    save_total_limit=2,
    #output_dir="grpo-testonly",
    remove_unused_columns=False,
    report_to=[],                     # отключаем wandb
    generation_kwargs={
        "max_new_tokens": 256,
        "temperature": 0.7,
        "top_p": 0.9,
        "do_sample": True,
    },
)

trainer_llm = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    args=grpo_config_ful,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    reward_funcs=reward_fn_combined,
)

trainer_llm.train()
#trainer_llm.save_model("deepseek-grpo-llmonly")
#llm_tokenizer.save_pretrained("deepseek-grpo-llmonly")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 32021}.


Step,Training Loss
10,0.000000
20,0.009800
30,-0.008900
40,0.018600
50,-0.010400
60,-0.001100
70,0.000300
80,0.000800
90,-0.003200
100,0.000800


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


TrainOutput(global_step=300, training_loss=3.4887343645095825e-05, metrics={'train_runtime': 3371.1556, 'train_samples_per_second': 0.089, 'train_steps_per_second': 0.089, 'total_flos': 0.0, 'train_loss': 3.4887343645095825e-05})

In [11]:
import ast
import textwrap

def _norm(value, v_min, v_max, invert=False):
    """Линейная нормализация в [0,1]. invert=True => меньшее значение лучше."""
    if v_max == v_min:
        return 1.0
    x = (value - v_min) / (v_max - v_min)
    x = max(0.0, min(1.0, x))
    if invert:
        x = 1.0 - x
    return float(x)

class _SimpleComplexityVisitor(ast.NodeVisitor):
    def __init__(self):
        self.cc = 1
        self.max_depth = 0
        self._cur_depth = 0

    def _enter(self):
        self.cc += 1
        self._cur_depth += 1
        self.max_depth = max(self.max_depth, self._cur_depth)

    def _exit(self):
        self._cur_depth -= 1

    def _branch(self, node):
        self._enter()
        self.generic_visit(node)
        self._exit()

    def visit_If(self, node): self._branch(node)
    def visit_For(self, node): self._branch(node)
    def visit_AsyncFor(self, node): self._branch(node)
    def visit_While(self, node): self._branch(node)
    def visit_Try(self, node): self._branch(node)
    def visit_With(self, node): self._branch(node)
    def visit_AsyncWith(self, node): self._branch(node)

    def visit_BoolOp(self, node): self._branch(node)
    def visit_IfExp(self, node): self._branch(node)

def _analyze_simple(code: str):
    """
    Возвращает:
      {
        "cc": ...,
        "nd": ...,
        "loc": ...,
        "comment_ratio": ...,
      }
    """
    code = textwrap.dedent(code)
    lines = code.splitlines()

    nonblank = [ln for ln in lines if ln.strip() != ""]
    comment_lines = [ln for ln in nonblank if ln.lstrip().startswith("#")]
    code_lines = [ln for ln in nonblank if not ln.lstrip().startswith("#")]

    loc = len(code_lines)
    total_nonblank = len(nonblank)
    comment_ratio = (
        len(comment_lines) / total_nonblank if total_nonblank > 0 else 0.0
    )

    try:
        tree = ast.parse(code)
    except SyntaxError:
        # считаем максимально сложным и без комментариев
        return {
            "cc": 20,
            "nd": 5,
            "loc": max(loc, 120),
            "comment_ratio": comment_ratio,
        }

    visitor = _SimpleComplexityVisitor()
    visitor.visit(tree)
    cc = visitor.cc
    nd = visitor.max_depth

    return {
        "cc": cc,
        "nd": nd,
        "loc": max(loc, 1),
        "comment_ratio": comment_ratio,
    }

def compute_static_reward_simple(code: str) -> float:
    """
    Упрощённый proxy static reward в [0,1] для коротких решений.
    """
    stats = _analyze_simple(code)

    cc = stats["cc"]
    nd = stats["nd"]
    loc = stats["loc"]
    comment_ratio = stats["comment_ratio"]

    # Границы нормализации под MBPP‑подобные задачи [web:210][web:180].
    s_cc  = _norm(cc,  1, 12, invert=True)   # 1–12 ветвлений
    s_nd  = _norm(nd,  0,  4, invert=True)   # до 4 уровней вложенности
    s_loc = _norm(loc, 3,  60, invert=True)  # 3–60 строк кода

    s_complexity = (s_cc + s_nd + s_loc) / 3.0
    s_readability = max(0.0, min(1.0, float(comment_ratio)))

    r_static = 0.6 * s_complexity + 0.4 * s_readability
    return max(0.0, min(1.0, float(r_static)))

In [12]:
@torch.no_grad()
def evaluate_model(
    model,
    tokenizer,
    dataset,
    max_new_tokens: int = 256,
):
    model.eval()
    task_results = []

    for ex in dataset:
        prompt    = ex["prompt"]
        test_code = ex["test_code"]
        task_id   = ex["task_id"]

        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        input_len = inputs["input_ids"].shape[1]

        gen_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

        completion = tokenizer.decode(
            gen_ids[0, input_len:], skip_special_tokens=True
        )
        code = extract_code(completion)

        # R_test
        passed, total = run_python_with_tests(code, test_code)
        r_test = passed / total if total > 0 else 0.0

        # R_static_simple
        r_static = compute_static_reward_simple(code)

        # комбинированная метрика
        if r_test+r_static==0:
            r_final_05=0
        else:
            r_final_05 = 2 * r_test * r_static / (r_test+r_static) 

        task_results.append(
            {
                "task_id": task_id,
                "r_test": r_test,
                "r_static": r_static,
                "r_final_05": r_final_05,
            }
        )

    mean_r_test = sum(r["r_test"] for r in task_results) / len(task_results)
    mean_r_static = sum(r["r_static"] for r in task_results) / len(task_results)
    mean_r_final_05 = sum(r["r_final_05"] for r in task_results) / len(task_results)

    return {
        "mean_r_test": mean_r_test,
        "mean_r_static": mean_r_static,
        "mean_r_final_05": mean_r_final_05,
        "per_task": task_results,
    }

In [16]:
del model
torch.cuda.empty_cache()

In [17]:
CKPT_PATH = "/kaggle/working/trainer_output/checkpoint-300"

tokenizer = AutoTokenizer.from_pretrained(CKPT_PATH, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    CKPT_PATH,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

In [18]:
evaluate_model(model, tokenizer, test_dataset)

{'mean_r_test': 0.35,
 'mean_r_static': 0.5346484497958187,
 'mean_r_final_05': 0.24854664575222427,
 'per_task': [{'task_id': 718,
   'r_test': 1.0,
   'r_static': 0.6,
   'r_final_05': 0.7499999999999999},
  {'task_id': 633, 'r_test': 0.0, 'r_static': 0.6, 'r_final_05': 0.0},
  {'task_id': 621,
   'r_test': 1.0,
   'r_static': 0.5318181818181817,
   'r_final_05': 0.6943620178041542},
  {'task_id': 524,
   'r_test': 0.0,
   'r_static': 0.6338424733161576,
   'r_final_05': 0.0},
  {'task_id': 478,
   'r_test': 1.0,
   'r_static': 0.6,
   'r_final_05': 0.7499999999999999},
  {'task_id': 874,
   'r_test': 0.0,
   'r_static': 0.5318181818181817,
   'r_final_05': 0.0},
  {'task_id': 867,
   'r_test': 0.0,
   'r_static': 0.7679197994987468,
   'r_final_05': 0.0},
  {'task_id': 532,
   'r_test': 1.0,
   'r_static': 0.6,
   'r_final_05': 0.7499999999999999},
  {'task_id': 357, 'r_test': 0.0, 'r_static': 0.6, 'r_final_05': 0.0},
  {'task_id': 207,
   'r_test': 0.0,
   'r_static': 0.33867623604

In [21]:
model.eval()
for ex in test_dataset:
    prompt    = ex["prompt"]
    test_code = ex["test_code"]
    task_id   = ex["task_id"]

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]

    gen_ids = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
    )

    # берём только сгенерированные токены после промпта
    gen_only = gen_ids[0, input_len:]
    completion = tokenizer.decode(gen_only, skip_special_tokens=True)
    code = extract_code(completion)
    print(f"PROMPT:\n{prompt}\n",f"CODE:\n{code}\n")

    # R_test
    passed, total = run_python_with_tests(code, test_code)
    r_test = passed / total if total > 0 else 0.0

    # R_LLM
    r_llm = call_local_judge(prompt, code)

    # комбинированная метрика для оценки
    r_final_05 = 0.5 * r_test + 0.5 * r_llm

    print(r_test, r_llm, r_final_05)

PROMPT:
You are an AI programming assistant, utilizing the
DeepSeek Coder model, developed by DeepSeek Company,
and you only answer questions related to computer science.

### Instruction:
Write a Python function that solves the following problem:
Write a function to create a list taking alternate elements from another given list.

Required function signature:
def alternate_elements(list1):

Requirements:
- Implement only the core function
- Include necessary import statements
- Ensure the function signature matches the test cases
- Write clean, efficient code

### Response:

 CODE:
def alternate_elements(list1):
    return [list1[i] for i in range(0, len(list1), 2)]


You are an expert code reviewer evaluating the quality of Python solutions.
Analyze the provided code based on the following criteria:

Problem Statement:
You are an AI programming assistant, utilizing the
DeepSeek Coder model, developed by DeepSeek Company,
and you only answer questions related to computer science.

###

KeyboardInterrupt: 